# IS 4487 Assignment 11: Predicting Airbnb Prices with Regression

In this assignment, you will:
- Load the Airbnb dataset you cleaned and transformed in Assignment 7
- Build a linear regression model to predict listing price
- Interpret which features most affect price
- Try to improve your model using only the most impactful predictors
- Practice explaining your findings to a business audience like a host, pricing strategist, or city partner

## Why This Matters

Pricing is one of the most important levers for hosts and Airbnb’s business teams. Understanding what drives price — and being able to predict it accurately — helps improve search results, revenue management, and guest satisfaction.

This assignment gives you hands-on practice turning a cleaned dataset into a predictive model. You’ll focus not just on code, but on what the results mean and how you’d communicate them to stakeholders.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_11_regression.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>



## Original Source: Dataset Description

The dataset you'll be using is a **detailed Airbnb listing file**, available from [Inside Airbnb](https://insideairbnb.com/get-the-data/).

Each row represents one property listing. The columns include:

- **Host attributes** (e.g., host ID, host name, host response time)
- **Listing details** (e.g., price, room type, minimum nights, availability)
- **Location data** (e.g., neighborhood, latitude/longitude)
- **Property characteristics** (e.g., number of bedrooms, amenities, accommodates)
- **Calendar/booking variables** (e.g., last review date, number of reviews)

The schema is consistent across cities, so you can expect similar columns regardless of the location you choose.

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score


## 1. Load Your Transformed Airbnb Dataset

**Business framing:**  
Before building any models, we must start with clean, prepared data. In Assignment 7, you exported a cleaned version of your Airbnb dataset. You’ll now import that file for analysis.

### Do the following:
- Import your CSV file called `cleaned_airbnb_data_7.csv`.   (Note: If you had significant errors with assignment 7, you can use the file named "airbnb_listings.csv" in the DataSets folder on GitHub as a backup starting point.)
- Use `pandas` to load and preview the dataset

### In Your Response:
1. What does the dataset include?
2. How many rows and columns are present?


In [4]:
import pandas as pd

# Load the cleaned dataset
try:
    df = pd.read_csv('cleaned_airbnb_data.csv')
except FileNotFoundError:
    print("cleaned_airbnb_data.csv not found. Please ensure the file is in the correct directory.")

# Preview the dataset
if 'df' in locals():
    display(df.head())

,id,listing_url,scrape_id,last_scraped,source,name,picture_url,host_id,host_url,host_name,...,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,44701,https://www.airbnb.com/rooms/44701,20250620182418,2025-06-20,city scrape,"Jan 4-11,2026 CES: 2beds,2baths, 1020 sqft,sle...",https://a0.muscache.com/pictures/1218626b-295f...,189245,https://www.airbnb.com/users/show/189245,Christine,...,5.00,5.00,4.75,4.50,f,2,2,0,0,0.04
1,113019,https://www.airbnb.com/rooms/113019,20250620182418,2025-06-21,city scrape,MGM Signature Studio !,https://a0.muscache.com/pictures/56149560/d180...,575684,https://www.airbnb.com/users/show/575684,LasVegasSuites,...,4.66,4.44,4.81,4.61,f,11,10,1,0,1.22
2,114140,https://www.airbnb.com/rooms/114140,20250620182418,2025-06-21,city scrape,MGM Signature 1BR/2BA Suite W/ View,https://a0.muscache.com/pictures/56068170/7fa6...,575684,https://www.airbnb.com/users/show/575684,LasVegasSuites,...,4.55,4.41,4.77,4.64,f,11,10,1,0,1.03
3,133084,https://www.airbnb.com/rooms/133084,20250620182418,2025-06-21,city scrape,"Beautiful, Sleek, Sexy Condo - Long Term Rental",https://a0.muscache.com/pictures/30c3a88e-3c4b...,653641,https://www.airbnb.com/users/show/653641,Brent,...,5.00,5.00,4.50,5.00,t,1,1,0,0,0.01
4,143096,https://www.airbnb.com/rooms/143096,20250620182418,2025-06-20,city scrape,Furnished Las Vegas House with Pool,https://a0.muscache.com/pictures/904198/63350b...,694506,https://www.airbnb.com/users/show/694506,Shari,...,4.93,4.89,4.77,4.70,f,1,1,0,0,1.49


1. The dataset includes information on Airbnb listings, covering host attributes, listing details, location data, property characteristics, and calendar/booking variables.
2. There are 17,891 rows and 73 columns.

## 2. Drop Columns Not Useful for Modeling

**Business framing:**  
Some columns — like post IDs or text — may not help us predict price and could add noise or bias.

### Do the following:
- Drop columns like `post_id`, `title`, `descr`, `details`, and `address` if they’re still in your dataset

### In Your Response:
1. What columns did you drop, and why?
2. What risks might occur if you included them in your model?


In [6]:
columns_to_drop = ['post_id', 'title', 'descr', 'details', 'address']
existing_columns_to_drop = [col for col in columns_to_drop if col in df.columns]

if existing_columns_to_drop:
    df = df.drop(columns=existing_columns_to_drop)
    print(f"Dropped columns: {existing_columns_to_drop}")
else:
    print("None of the specified columns were found in the DataFrame.")

# Display the updated columns
print("Remaining columns:")
display(df.columns)

None of the specified columns were found in the DataFrame.
Remaining columns:


Index(['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name',
       'picture_url', 'host_id', 'host_url', 'host_name', 'host_since',
       'host_location', 'host_response_time', 'host_response_rate',
       'host_acceptance_rate', 'host_is_superhost', 'host_thumbnail_url',
       'host_picture_url', 'host_neighbourhood', 'host_listings_count',
       'host_total_listings_count', 'host_verifications',
       'host_has_profile_pic', 'host_identity_verified', 'neighbourhood',
       'neighbourhood_cleansed', 'latitude', 'longitude', 'property_type',
       'room_type', 'accommodates', 'bathrooms', 'bathrooms_text', 'bedrooms',
       'beds', 'amenities', 'price', 'minimum_nights', 'maximum_nights',
       'minimum_minimum_nights', 'maximum_minimum_nights',
       'minimum_maximum_nights', 'maximum_maximum_nights',
       'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm', 'has_availability',
       'availability_30', 'availability_60', 'availability_90',
       'availabilit

1. None of the columns listed were in the dataset.
2. Including columns like these would not be useful for a linear regression model. Including them could lead to a less accurate model.

## 3. Explore Relationships Between Numeric Features

**Business framing:**  
Understanding how features relate to each other — and to the target — helps guide feature selection and modeling.

### Do the following:
- Generate a correlation matrix
- Identify which variables are strongly related to `price`

### In Your Response:
1. Which variables had the strongest positive or negative correlation with price?
2. Which variables might be useful predictors?


In [7]:
# Generate the correlation matrix
correlation_matrix = df.corr(numeric_only=True)

# Display the correlation with 'price'
display(correlation_matrix['price'].sort_values(ascending=False))

,price
price,1.000000
host_total_listings_count,0.401880
estimated_revenue_l365d,0.209993
calculated_host_listings_count,0.144523
calculated_host_listings_count_private_rooms,0.134511
host_listings_count,0.115274
maximum_nights,0.093948
availability_30,0.065161
bathrooms,0.052594
availability_60,0.049099


1. The variables with the strongest positive correlation to price are host total listings count, estimated revenue l365d, and calculatede host listings count. The variables with the strongest negative correlation to price are review scores accuracy, estimated occupancy l365d, and review scores value.
2. Variables with a stronger absolute correlation with price might be more useful because they show a stronger linear connection with price.

## 4. Define Features and Target Variable

**Business framing:**  
To build a regression model, you need to define what you’re predicting (target) and what you’re using to make that prediction (features).

### Do the following:
- Set `price` as your target variable
- Remove `price` from your predictors

### In Your Response:
1. What features are you using?
2. Why is this a regression problem and not a classification problem?


In [8]:
# Define the target variable
y = df['price']

# Define the features by dropping the target variable
X = df.drop('price', axis=1)

1. The features I am using are all the columns in the DataFrame except for the price column.
2. This is a regression problem because we are trying to predict a continuous numerical value, the price of an Airbnb listing.

## 5. Split Data into Training and Testing Sets

### Business framing:
Splitting your data lets you train a model and test how well it performs on new, unseen data.

### Do the following:
- Use `train_test_split()` to split into 80% training, 20% testing



In [9]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Data split into training and testing sets:")
print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")
print(f"Training target shape: {y_train.shape}")
print(f"Testing target shape: {y_test.shape}")

Data split into training and testing sets:
Training features shape: (14312, 72)
Testing features shape: (3579, 72)
Training target shape: (14312,)
Testing target shape: (3579,)


## 6. Fit a Linear Regression Model

### Business framing:
Linear regression helps you quantify the impact of each feature on price and make predictions for new listings.

### Do the following:
- Fit a linear regression model to your training data
- Use it to predict prices for the test set



In [18]:
# Initialize and fit the Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Predict prices on the test set
y_pred = model.predict(X_test)

## 7. Evaluate Model Performance

### Business framing:  
A good model should make accurate predictions. We’ll use Mean Squared Error (MSE) and R² to evaluate how close our predictions were to the actual prices.

### Do the following:
- Print MSE and R² score for your model

### In Your Response:
1. What is your R² score? How well does your model explain price variation?
2. Is your MSE large or small? What could you do to improve it?


In [19]:
# Evaluate model performance
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error (MSE): {mse}")
print(f"R-squared (R²): {r2}")

Mean Squared Error (MSE): 3941432.644239035
R-squared (R²): 0.00023603839016439476


1. My r-squared score is 0.000236. This is very close to 0, and is not a good predictor for the target variable.
2. My MSE is far too large, and some things that I could do to improve it would be to use one-hot encoding and handle outliers better.

## 8. Interpret Model Coefficients

### Business framing:
The regression coefficients tell you how each feature impacts price. This can help Airbnb guide hosts and partners.

### Do the following:
- Create a table showing feature names and regression coefficients
- Sort the table so that the most impactful features are at the top

### In Your Response:
1. Which features increased price the most?
2. Were any surprisingly negative?
3. What business insight could you draw from this?


In [21]:
# Get the feature names and coefficients
coefficients = pd.DataFrame({
    'feature': X_train.columns,
    'coefficient': model.coef_
})

# Calculate the absolute value of coefficients for sorting
coefficients['abs_coefficient'] = abs(coefficients['coefficient'])

# Sort by absolute coefficient in descending order
coefficients = coefficients.sort_values(by='abs_coefficient', ascending=False)

# Display the sorted coefficients (excluding the absolute value column)
display(coefficients[['feature', 'coefficient']])

,feature,coefficient
2,host_id,5.900343e-08
15,maximum_maximum_nights,-4.399714e-08
17,maximum_nights_avg_ntm,-4.387660e-08
28,estimated_revenue_l365d,4.093113e-13
14,minimum_maximum_nights,1.673214e-14
11,maximum_nights,1.257153e-14
4,host_total_listings_count,-4.892557e-15
21,availability_365,4.200724e-15
37,calculated_host_listings_count_entire_homes,-2.766523e-15
36,calculated_host_listings_count,-2.338733e-15


1. Based on the coefficients, host id, estimated revenue l365d, and minimum maximum nights had the largest positive coefficients. This model does not clearly show which features increased price the most.

2. Several features had negative coefficients, including various review scores and host listing counts, but the very small coefficient values make it difficult to identify.

3. The main business insight is that this linear model, using only the numeric features, does not provide clear guidance on which factors significantly impact Airbnb prices; further analysis with different features or models is needed.

## 9. Try to Improve the Linear Regression Model

### Business framing:
The first version of your model included all available features — but not all features are equally useful. Removing weak or noisy predictors can often improve performance and interpretation.

### Do the following:
1. Choose your top 3–5 features with the strongest absolute coefficients
2. Rebuild the regression model using just those features
3. Compare MSE and R² between the baseline and refined model

### In Your Response:
1. What features did you keep in the refined model, and why?
2. Did model performance improve? Why or why not?
3. Which model would you recommend to stakeholders?
4. How does this relate to your customized learning outcome you created in canvas?


In [22]:
# Choose top features based on absolute coefficients
# Let's select the top 5 features
top_features = coefficients.head(5)['feature'].tolist()

print(f"Top 5 features selected: {top_features}")

# Create new feature sets with only the top features
X_train_refined = X_train[top_features]
X_test_refined = X_test[top_features]

# Initialize and fit the Linear Regression model with refined features
model_refined = LinearRegression()
model_refined.fit(X_train_refined, y_train)

# Predict prices on the test set with the refined model
y_pred_refined = model_refined.predict(X_test_refined)

# Evaluate refined model performance
mse_refined = mean_squared_error(y_test, y_pred_refined)
r2_refined = r2_score(y_test, y_pred_refined)

print("\nRefined Model Performance:")
print(f"Mean Squared Error (MSE): {mse_refined}")
print(f"R-squared (R²): {r2_refined}")

print("\nBaseline Model Performance:")
print(f"Mean Squared Error (MSE): {mse}")
print(f"R-squared (R²): {r2}")

Top 5 features selected: ['host_id', 'maximum_maximum_nights', 'maximum_nights_avg_ntm', 'estimated_revenue_l365d', 'minimum_maximum_nights']

Refined Model Performance:
Mean Squared Error (MSE): 3186696.9756187084
R-squared (R²): 0.19167848841680613

Baseline Model Performance:
Mean Squared Error (MSE): 3941432.644239035
R-squared (R²): 0.00023603839016439476


1. I kept the top 5 features with the strongest absolute coefficients from the initial model: host id, maximum maximum nights, maximum nights avg ntm, estimated revenue l365d, and minimum maximum nights.

2. Yes, the refined model's performance improved significantly. The R² score increased from a very low 0.0002 to 0.1917, and the MSE decreased from over 3.9 million to about 3.1 million.

3. Based on the improved R² and lower MSE, I would recommend the refined model to stakeholders over the baseline model. While the refined model still doesn't explain a large proportion of the price variation, it is a better predictor than the initial model that included all numeric features.

## 10. Reflect and Recommend

### Business framing:  
Ultimately, the value of your model comes from how well it can guide business decisions. Use your results to make real-world recommendations.

### In Your Response:
1. What business question did your model help answer?
2. What would you recommend to Airbnb or its hosts?
3. What could you do next to improve this model or make it more useful?
4. How does this relate to your customized learning outcome you created in canvas?


1.  This model helped explore which numeric features had the strongest linear connection to Airbnb price. It showed that the numeric features in this dataset have a weak linear relationship with price.

2.  I'd recommend to Airbnb and hosts that focusing only on these numeric factors probably won't change prices much. Other things, likely non-numeric details, seem to matter more for pricing.

3.  To make the model better, we should add and encode non-numeric features like room type and location. We could also try different model types that handle relationships better than simple linear regression.

4.  This assignment helped me practice building and evaluating a prediction model, and explaining what the results mean, connecting directly to my learning outcome of interpreting data.

In [23]:
!jupyter nbconvert --to html "assignment_11_BensonDallin.ipynb"

[NbConvertApp] Converting notebook assignment_11_BensonDallin.ipynb to html
[NbConvertApp] Writing 340312 bytes to assignment_11_BensonDallin.html
